# فصل 9: بینایی ماشین در GeoAI

کد این نوت‌بوک عیناً از فصل 9 کتاب «مهندسی هوش مصنوعی مکانی» برداشته شده است.

پیش از اجرا: مسیر داده‌ها را با ساختار پوشه‌ی `data/`، `work/` و `out/` هماهنگ کنید. راه دانلود هر فایل در پیوست ب کتاب آمده است.


## ۹-۲) کارگاه ۱: استخراج خودکار پلیگون ساختمان با U-Net


In [ ]:
!pip install segmentation_models tensorflow rasterio geopandas

import segmentation_models as sm
import numpy as np, rasterio, geopandas as gpd
from rasterio.features import shapes
from shapely.geometry import shape

sm.set_framework("tf.keras")
model = sm.Unet(
    backbone_name="resnet34",      # ResNet34 pre-trained on ImageNet
    encoder_weights="imagenet",
    classes=1,                     # one class: building
    activation="sigmoid",
)

In [ ]:
import tensorflow as tf

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=sm.losses.bce_jaccard_loss,   # cross-entropy + Jaccard together
    metrics=[sm.metrics.iou_score],    # use IoU, NOT accuracy
)

history = model.fit(X_train, Y_train, validation_split=0.2,
                    epochs=50, batch_size=8)

In [ ]:
with rasterio.open("new_area.tif") as src:
    img = src.read().transpose(1, 2, 0) / 255.0   # CHW -> HWC, normalise
    transform, crs = src.transform, src.crs

mask = model.predict(np.expand_dims(img, 0))[0, :, :, 0]
binary = (mask > 0.5).astype("uint8")

# turn the pixel mask into geo-referenced polygons
polys = [shape(geom)
         for geom, val in shapes(binary, transform=transform)
         if val == 1]

gdf = gpd.GeoDataFrame(geometry=polys, crs=crs)
gdf.to_file("buildings_extracted.shp")
print("buildings found:", len(gdf))

## ۹-۳) کارگاه ۲: کشف استخرهای ثبت‌نشده با YOLO


In [ ]:
!pip install ultralytics

from ultralytics import YOLO

# dataset.yaml describes the data layout:
#   path:  /content/pools_data
#   train: images/train
#   val:   images/val
#   nc:    1
#   names: ["pool"]
# each label .txt holds: class_id x_center y_center width height  (all 0..1)

In [ ]:
model = YOLO("yolov8n.pt")        # nano: small enough for a laptop
model.train(data="pools_data/dataset.yaml", epochs=50, imgsz=640, batch=16)
results = model.predict(source="city_aerial.tif", conf=0.4, save=True)
print("pools found:", len(results[0].boxes))

In [ ]:
import rasterio, geopandas as gpd
from shapely.geometry import Point

with rasterio.open("city_aerial.tif") as src:
    transform, crs = src.transform, src.crs

pts = []
for box in results[0].boxes:
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    xc, yc = (x1 + x2) / 2, (y1 + y2) / 2      # box centre, in pixels
    x_geo, y_geo = rasterio.transform.xy(transform, yc, xc)
    pts.append({"geometry": Point(x_geo, y_geo),
                "confidence": float(box.conf[0])})

gpd.GeoDataFrame(pts, crs=crs).to_file("pools_detected.shp")

## واقعیت ۱: ابعاد غول‌پیکر (کاشی‌بندی)


In [ ]:
def tile_image(image, tile=512, overlap=64):
    h, w = image.shape[:2]
    step = tile - overlap
    tiles, positions = [], []
    for y in range(0, h - overlap, step):
        for x in range(0, w - overlap, step):
            y2, x2 = min(y + tile, h), min(x + tile, w)
            tiles.append(image[y:y2, x:x2])
            positions.append((y, x))
    return tiles, positions   # predict each, then stitch back

## واقعیت ۳: نبود بالا و پایین (مقاومت به چرخش)


In [ ]:
import albumentations as A
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
])